## Exploratory Analysis of Fraud Transactions

This notebook performs **exploratory data analysis (EDA)** on the full fraud transaction dataset (21 million records) to understand **class imbalance, fraud patterns, and limitations of the provided fraud flag**.

The analysis examines fraud prevalence, transaction types, monetary impact, temporal patterns (by hour and day), and repeated account involvement.

These insights are used to motivate subsequent stages of the project, where **supervised and unsupervised machine learning models** are applied to learn fraud risk directly from transaction data using a **stratified 5M-row sample** for modelling efficiency.


## 1. Setup Environment and Load the Dataset

In [ ]:
#mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#import required libraries
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

from datasets import load_dataset

In [ ]:
#load the dataset
ds = load_dataset('CiferAI/Cifer-Fraud-Detection-Dataset-AF')

# 2.Exploratory Data Analysis

In [ ]:
#inspect dataset dimensions
ds.shape

In [ ]:
#convert the loaded data to pandas DataFrame
df=ds['train'].to_pandas()

In [ ]:
#explore unique transaction types
df['type'].unique()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df['isFraud'].value_counts(normalize=True)

In [ ]:
df['isFlaggedFraud'].value_counts(normalize=True)

In [ ]:
#num of frauds caught by the flag
caught_fraud=df[(df['isFraud']==1) & (df['isFlaggedFraud']==1)].shape[0]
print(f"Number of frauds caught by flag:",caught_fraud)

#total num of frauds
total_fraud=df['isFraud'].sum()
print(f"Total number of frauds:",total_fraud)


#proportion caught
proportion_caught=caught_fraud/total_fraud
print(f"Proportion of frauds caught:",proportion_caught)

In [ ]:
#total counts
total_transactions=len(df)

total_fraud=df['isFraud'].sum()
#total_flagged=df['isFlaggedFraud'].sum()
total_flagged_fraud=(df['isFlaggedFraud']==1).sum()
actual_flagged_fraud=df[(df['isFraud']==1) & (df['isFlaggedFraud']==1)].shape[0]

#summary table
summary=pd.DataFrame({
    'Metric':['Total Transactions','Total Fraud','Total Flagged Fraud','Actual Flagged Fraud'],
    'Count':[total_transactions,total_fraud,total_flagged_fraud, actual_flagged_fraud],
    'Porportion(%)': [100, total_fraud/total_transactions*100, total_flagged_fraud/total_transactions*100, actual_flagged_fraud/total_transactions*100]
})

print(summary)

**Notes:**

**Class Imbalance**

- Only **approximately 0.13%** of transactions in the full dataset are fraudulent, while **approximately 99.87%** are legitimate.  

- This reflects **extreme class imbalance**, which is typical in real-world fraud detection problems.


**Flagged Transactions in the Dataset**

- Only **approximately 0.0003%** of transactions are flagged as fraud in the dataset.  

- Almost all transactions are unflagged.  
Compared to the actual fraud rate (approximately 0.13%), the dataset's fraud flag misses the vast majority of fraudulent transactions.


In [ ]:
#metrics
metrics=['Total Transactions','Total Fraud','Total Flagged Fraud','Actual Flagged Fraud']
counts=[total_transactions,total_fraud,total_flagged_fraud, actual_flagged_fraud]

#plot bar chart
plt.figure(figsize=(10,6))
bars=plt.bar(metrics, counts, color=['skyblue','salmon','pink','orange'])
plt.title('Fraud Analysis Overview')
plt.ylabel('Count of Transactions (log scale)')
plt.yscale('log')
plt.xticks(rotation=15)

#annotate bars
for bar in bars:
  yval=bar.get_height()
  plt.text(bar.get_x()+bar.get_width()/2, yval, f'{yval:,}', ha='center', va='bottom', fontsize=10)

plt.show()

In [ ]:
#summarise fraud and flagged fraud statistics by transaction type
table=df.groupby('type').agg(
    fraud_count=('isFraud', 'sum'),
    flagged_count=('isFlaggedFraud', 'sum'),
    fraud_and_flagged=('isFraud', lambda x: ((x==1) & (df.loc[x.index, 'isFlaggedFraud']==1)).sum()),
    total=('isFraud', 'count')
)

table['fraud_percent'] = table['fraud_count'] / table['total'] * 100
table['flagged_percent'] = table['flagged_count'] / table['total'] * 100

print(table)

**Notes:**

- Fraud occurs across ALL types.
- DEBIT transactions are few, but proportionally riskier.
- CASH_OUT and PAYMENT represent the largest share of total fraud volume, because they are used heavily and fraudsters prefer them for easy cash exit.


In [ ]:
df.groupby('isFraud')['amount'].describe()

In [ ]:
#calculate total transaction amount involved in actual fraud

total_fraud_amount=df.loc[df['isFraud']==1, 'amount'].sum()
print('Total transaction amount of actual fraud:', total_fraud_amount)

In [ ]:
#calculate total transaction amount of correctly flagged fraud

total_flagged_fraud_amount=df.loc[(df['isFlaggedFraud']==1) & (df['isFraud']==1), 'amount'].sum()
print('Total transaction amount of flagged fraud:', total_flagged_fraud_amount)

In [ ]:
#analyse fraud amount by transaction type

fraud_per_type=df[df['isFraud']==1].groupby('type')['amount'].sum()
print(fraud_per_type)

**Notes:**

Only **approximately 0.23%** of the total fraud value (about \$11.7M out of about \$5.1B) is captured by the dataset's fraud flag, meaning **around 99.77% of fraud losses are not reflected in the label**.

This shows that the flag misses not only most fraud cases, but also the **monetary impact** of fraud.


In [ ]:
#derive day and hour features from the 'step' feature
df['day']=(df['step']//24)+1
df['hour']=df['step']%24

In [ ]:
#visualise the number of fraudulent transactions per hour

fraud_per_hour=df.groupby('hour')['isFraud'].sum()
fraud_per_hour

plt.figure(figsize=(10,5))
fraud_per_hour.plot(kind='bar')
plt.xlabel('Hour of the Day(0-23)')
plt.ylabel('Number of Fraudulent Transactions')
plt.title('Number of Fraudulent Transactions per Hour')
plt.show()

**Notes:**
- Fraud happens consistently throughout the 24 hrs.
- Small peaks around:
  - 22:00 (highest)
  - 08:00-9:00
- Slight dips during:
12:00, 14:00, 15:00 (working hours)
05:00 (early morning)

In [ ]:
#visualise the number of fraudulent transactions per day
fraud_per_day=df.groupby('day')['isFraud'].sum()

plt.figure(figsize=(10,5))
fraud_per_day.plot(kind='bar')
plt.title('Number of Fraudlent Transaction per day')
plt.xlabel('Day (1-31)')
plt.ylabel('Number of Fraudulent Transactions')
plt.show()

**Notes:**
- Fraud peaks on days 5-9 (early month).
- Gradual decline after day 10.
- Lowest activity around days 25-30.
- Sudden small rebound on day 31.

In [ ]:
#summarise and visualise transaction counts by hour and fraud status
hourly_stats=df.groupby('hour').agg(
    total_transaction=('amount', 'count'),
    non_fraud=('isFraud', lambda x: (x==0).sum()),
    actual_fraud=('isFraud', 'sum'),
    flagged_fraud=('isFlaggedFraud', 'sum')
)

hourly_stats.plot(kind='bar', figsize=(12,6))
plt.title('Transactions by Hour: Total, Non-Fraud, Actual Fraud, Flagged Fraud')
plt.xlabel('Hour of Day (0-23)')
plt.ylabel('Count of Transactions')
plt.xticks(rotation=0)
plt.legend(loc='upper right')
plt.show()

In [ ]:
#summarise and visualise daily transaction counts by fraud status

daily_stats=df.groupby('day').agg(
    total_transaction=('amount', 'count'),
    non_fraud=('isFraud', lambda x: (x==0).sum()),
    actual_fraud=('isFraud', 'sum'),
    flagged_fraud=('isFlaggedFraud', 'sum')
)
daily_stats.plot(kind='bar', figsize=(12,6))
plt.title('Transactions by Day: Total, Non-Fraud, Actual Fraud, Flagged Fraud')
plt.xlabel('Day of Month (1-31)')
plt.ylabel('Count of Transactions')
plt.xticks(rotation=0)
plt.legend(loc='upper right')
plt.show()

In [ ]:
#select only transactions that are actual fraud
fraud_df=df[df['isFraud']==1]

#count num of times each origin account appear in actual fraud
original_counts=fraud_df['nameOrig'].value_counts()

#count num of times each destination account appear in actual fraud
dest_counts=fraud_df['nameDest'].value_counts()

#find accounts involved in fraud more than once
repeated_origins=original_counts[original_counts>1]
repeated_dests=dest_counts[dest_counts>1]

#print sender and receiver accounts with repeated fraud activity
print('Original accounts involved in multiple frauds:')
print(repeated_origins)

print('Destination accounts involved in multiple frauds:')
print(repeated_dests)

In [ ]:
#print the number of accounts repeatedly involved in fraud

print(f'Number of original accounts involved in actual fraud:', len(repeated_origins))
print(f'\nNumber of original accounts involved in actual fraud:', len(repeated_dests))